In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated
from dotenv import load_dotenv, find_dotenv

In [ ]:
load_dotenv(find_dotenv())

model = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.7)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat(state: ChatState):
    response = model.invoke(state['messages'])
    return {"messages": [response]}    # model return in AIMessage format, no need to wrap it again inside it

In [ ]:
graph = StateGraph(ChatState)

graph.add_node("chat", chat)

graph.add_edge(START, "chat")
graph.add_edge("chat", END)


memory_checkpointer = InMemorySaver()

chatbot = graph.compile(checkpointer=memory_checkpointer)

In [ ]:
thread_id = "user1"
config_user_a = {"configurable":{"thread_id":thread_id}}

while True:
    query = input()

    if query.strip().lower() in ["exit", "quit", "bye"]:
        break

    print("User:", query)

    response = chatbot.invoke({"messages": [HumanMessage(query)]}, config=config_user_a)

    print("AI:", response["messages"][-1].content)


In [ ]:
chatbot.get_state(config=config_user_a)